<a href="https://colab.research.google.com/github/Lucaschewitch/Machine-Learning/blob/main/%D0%9F%D0%A013.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практическая работа. Геоанализ экологических факторов городской среды с применением методов пространственной кластеризации**




















## **Цель работы**



Освоение методов пространственного анализа и машинного обучения для оценки экологической обстановки городской территории с использованием гексагональной сетки H3, с последующим применением различных алгоритмов кластеризации и классификации.

## **Задачи**



1. Освоить инструменты загрузки и агрегации экологических геоданных с использованием API OpenStreetMap
2. Реализовать анализ пространственного распределения экологических факторов с помощью гексагональной сетки H3
3. Применить и сравнить различные алгоритмы кластеризации для выявления однородных экологических зон
4. Обучить модели классификации для прогнозирования экологического состояния новых территорий

## **Теоретическая часть**



Современный геоэкологический анализ требует комплексного подхода к обработке пространственно-распределенных данных. Применение методов машинного обучения, в частности алгоритмов кластеризации (K-means, агломеративной, спектральной, DBSCAN, HDBSCAN), позволяет выявлять неявные закономерности в распределении экологических факторов городской среды и определять территории со схожими экологическими характеристиками.

## **Этапы работы**



### **1. Определение области исследования и подготовка данных**


- Выберите городскую территорию для анализа экологической обстановки
- Используя API OpenStreetMap, загрузите данные следующих категорий:
  - Источники загрязнения (промышленные предприятия, мусоропереработка, ТЭЦ)
  - Зеленые насаждения (парки, скверы, лесопарковые зоны)
  - Водные объекты (реки, водоемы)
  - Автомагистрали (категории дорог с интенсивным движением)
  - Административные районы города

In [1]:
# Ваш код
!pip install scikit-learn geopandas h3pandas h3~=3.0 leafmap mapclassify matplotlib osmnx kneed hdbscan joblib -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.4/138.4 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 667.5/667.5 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 110.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import h3
import leafmap
from shapely.geometry import box, Polygon
from sklearn.preprocessing import MinMaxScaler
import osmnx as ox
import warnings
warnings.filterwarnings('ignore')

In [3]:
m = leafmap.Map(draw_control=True, basemap='CartoDB.Positron')
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

In [4]:
bbox = m.user_roi_bounds()
if bbox is None:
    bbox = [37.85, 55.85, 38.15, 56.05]
    print(f"Используем bbox по умолчанию: {bbox}")
print(bbox)

[37.9552, 55.9059, 38.0393, 55.9379]


In [5]:
bbox = m.user_roi_bounds()
if bbox is None:
    bbox = [37.85, 55.85, 38.15, 56.05]
    print(f"Используем bbox по умолчанию: {bbox}")
print(bbox)

[37.9552, 55.9059, 38.0393, 55.9379]


In [6]:
def load_osm(bbox):
    west, south, east, north = bbox
    tags = {
        'poll': {'landuse': ['industrial', 'landfill'], 'power': 'plant', 'man_made': 'works'},
        'green': {'leisure': ['park', 'garden'], 'natural': 'wood', 'landuse': 'forest'},
        'water': {'waterway': 'river', 'natural': 'water', 'water': ['lake', 'reservoir', 'pond']},
        'road': {'highway': ['primary', 'trunk', 'motorway', 'secondary', 'tertiary']}
    }
    out = {}
    for k, t in tags.items():
        try:
            g = ox.features_from_bbox((west, south, east, north), tags=t)
            out[k] = g
        except:
            out[k] = gpd.GeoDataFrame(columns=['geometry'])
    return out

In [7]:
data = load_osm(bbox)
crs_proj = "EPSG:3857"
for k in data:
    if not data[k].empty and 'geometry' in data[k].columns:
        data[k] = data[k].to_crs(crs_proj)
    else:
        data[k] = gpd.GeoDataFrame(columns=['geometry'], crs=crs_proj)

### **2. Агрегация данных с использованием гексагональной сетки H3**


- Сгенерируйте гексагональную сетку H3 оптимального разрешения для выбранной территории
- Для каждой ячейки H3 рассчитайте:
  - Количество и плотность источников загрязнения в радиусе 1600м
  - Площадь и процент покрытия зелеными насаждениями в радиусе 800м
  - Протяженность водных объектов в радиусе 1600м
  - Плотность автомагистралей в радиусе 1600м
- Выполните нормализацию полученных показателей
- Сформируйте интегральный индекс экологического благополучия территории


In [8]:
# Ваш код
def calc_cell(geom):
    b1600 = geom.buffer(1600)
    b800 = geom.buffer(800)
    n_poll = data['poll'].geometry.intersects(b1600).sum()
    len_water = data['water'].geometry.intersection(b1600).length.sum()
    len_road = data['road'].geometry.intersection(b1600).length.sum()
    area_green = data['green'].geometry.intersection(b800).area.sum()
    return n_poll, len_water, len_road, area_green

In [31]:
bbox_poly = box(*bbox)
bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_poly], crs="EPSG:4326").to_crs(crs_proj)
res = 9
h3_idx = list(h3.polyfill(bbox_poly.__geo_interface__, res))
h3_geom = [Polygon(h3.h3_to_geo_boundary(h, geo_json=True)) for h in h3_idx]
h3_gdf = gpd.GeoDataFrame({'h3': h3_idx, 'geometry': h3_geom}, crs="EPSG:4326").to_crs(crs_proj)

In [32]:
feat = ['n_poll', 'len_water', 'len_road', 'area_green']
h3_gdf[feat] = h3_gdf.geometry.apply(lambda g: pd.Series(calc_cell(g)))

In [33]:
scaler = MinMaxScaler()
h3_gdf[feat] = scaler.fit_transform(h3_gdf[feat])
h3_gdf['score'] = h3_gdf['len_water'] + h3_gdf['area_green'] - h3_gdf['n_poll'] - h3_gdf['len_road']

In [34]:
X = h3_gdf[feat + ['score']].copy()
X_scaled = scaler.fit_transform(X)

### **3. Определение оптимального числа кластеров**


- Постройте график метода локтя (Elbow method) для определения оптимального числа кластеров
- Оцените качество кластеризации с помощью внутренних метрик кластеризации
- Определите оптимальное число кластеров на основе комбинации различных метрик

In [35]:
# Ваш код
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import matplotlib.pyplot as plt
from kneed import KneeLocator

In [36]:
inert = []
sil = []
cal = []
dav = []
k_range = range(2,13)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inert.append(km.inertia_)
    sil.append(silhouette_score(X_scaled, km.labels_))
    cal.append(calinski_harabasz_score(X_scaled, km.labels_))
    dav.append(davies_bouldin_score(X_scaled, km.labels_))

ValueError: Number of labels is 1. Valid values are 2 to n_samples - 1 (inclusive)

In [ ]:
fig, ax = plt.subplots(2,2,figsize=(12,10))
ax[0,0].plot(k_range, inert, 'o-')
ax[0,0].set_title('Inertia')
ax[0,1].plot(k_range, sil, 'o-')
ax[0,1].set_title('Silhouette')
ax[1,0].plot(k_range, cal, 'o-')
ax[1,0].set_title('Calinski')
ax[1,1].plot(k_range, dav, 'o-')
ax[1,1].set_title('Davies')
plt.tight_layout()
plt.show()

In [ ]:
kl = KneeLocator(k_range, inert, curve='convex', direction='decreasing')
print("Elbow k:", kl.elbow)
print("Max Silhouette k:", k_range[np.argmax(sil)])

In [ ]:
from sklearn.neighbors import NearestNeighbors

In [ ]:
nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_scaled)
dist, _ = nn.kneighbors(X_scaled)
dist = np.sort(dist[:,4])
kl2 = KneeLocator(range(len(dist)), dist, curve='convex', direction='increasing')
eps_opt = dist[kl2.knee] if kl2.knee else np.percentile(dist, 10)
print("eps:", eps_opt)

### **4. Сравнительный анализ алгоритмов кластеризации**


- Реализуйте и сравните следующие алгоритмы кластеризации:
  - K-means
  - Агломеративная кластеризация
  - Спектральная кластеризация
  - DBSCAN (с оптимальным значением eps)
  - HDBSCAN
- Для каждого алгоритма оцените:
  - Качество кластеризации по основным метрикам
  - Распределение точек по кластерам
  - Характерные особенности выявленных кластеров

In [ ]:
# Ваш код
from sklearn.cluster import AgglomerativeClustering, SpectralClustering, DBSCAN
import hdbscan

In [ ]:
def eval_clust(X, labels, name):
    uniq = np.unique(labels)
    n_clust = len(uniq) - (1 if -1 in uniq else 0)
    if n_clust < 2:
        return {'sil':np.nan, 'cal':np.nan, 'dav':np.nan, 'n':n_clust}
    return {
        'sil': silhouette_score(X, labels),
        'cal': calinski_harabasz_score(X, labels),
        'dav': davies_bouldin_score(X, labels),
        'n': n_clust
    }

In [ ]:
res = {}
k_opt = [3,4,5]
for k in k_opt:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    res[f'KMeans_{k}'] = eval_clust(X_scaled, km.fit_predict(X_scaled), '')

    ag = AgglomerativeClustering(n_clusters=k)
    res[f'Agglo_{k}'] = eval_clust(X_scaled, ag.fit_predict(X_scaled), '')

    sp = SpectralClustering(n_clusters=k, random_state=42, assign_labels='kmeans')
    res[f'Spectral_{k}'] = eval_clust(X_scaled, sp.fit_predict(X_scaled), '')

In [ ]:
db = DBSCAN(eps=eps_opt)
res['DBSCAN'] = eval_clust(X_scaled, db.fit_predict(X_scaled), '')

hdb = hdbscan.HDBSCAN()
res['HDBSCAN'] = eval_clust(X_scaled, hdb.fit_predict(X_scaled), '')

In [ ]:
pd.DataFrame(res).T.sort_values('sil', ascending=False)

In [ ]:
best = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = best.fit_predict(X_scaled)
h3_gdf['cluster'] = labels

### **5. Визуализация и интерпретация результатов**


- Визуализируйте результаты кластеризации на карте с использованием leafmap
- Постройте тепловые карты средних значений экологических факторов для каждого кластера
- Выполните снижение размерности с помощью PCA и визуализируйте кластеры в двумерном пространстве
- Проанализируйте профили кластеров и составьте их экологические характеристики
- Разработайте рекомендации по улучшению экологической обстановки для каждого типа территории

In [ ]:
# Ваш код
h3_viz = h3_gdf.to_crs("EPSG:4326")
m2 = leafmap.Map(center=[(bbox[1]+bbox[3])/2, (bbox[0]+bbox[2])/2], zoom=12, basemap='CartoDB.Positron')
m2.add_data(h3_viz, column='cluster', cmap='Set1', legend_title='Кластер', style={'fillOpacity':0.7, 'weight':0})
m2

In [ ]:
import seaborn as sns
means = []
for c in sorted(h3_gdf['cluster'].unique()):
    means.append(h3_gdf[h3_gdf['cluster']==c][feat+['score']].mean())
df_means = pd.DataFrame(means, index=[f'Кл{c}' for c in sorted(h3_gdf['cluster'].unique())], columns=feat+['score'])
plt.figure(figsize=(10,6))
sns.heatmap(df_means, annot=True, cmap='coolwarm', fmt='.3f')
plt.title('Средние значения признаков по кластерам')
plt.show()

In [ ]:
from sklearn.decomposition import PCA
import plotly.express as px
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
fig = px.scatter(x=X_pca[:,0], y=X_pca[:,1], color=labels.astype(str), title='PCA проекция кластеров')
fig.show()

### **6. Разработка моделей классификации**


- На основе результатов лучшего алгоритма кластеризации подготовьте данные для обучения классификаторов
- Обучите и сравните различные модели классификации:
  - Логистическая регрессия
  - Дерево решений
  - Случайный лес
  - Градиентный бустинг
  - SVM
  - K-ближайших соседей
- Оцените качество моделей с использованием кросс-валидации
- Выберите оптимальную модель и сохраните её для дальнейшего использования

In [ ]:
# Ваш код
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, labels, test_size=0.3, random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, f1_score

In [ ]:
models = {
    'LR': LogisticRegression(),
    'Tree': DecisionTreeClassifier(),
    'RF': RandomForestClassifier(),
    'GB': GradientBoostingClassifier(),
    'SVM': SVC(),
    'KNN': KNeighborsClassifier()
}

In [ ]:
for name, m in models.items():
    m.fit(X_tr, y_tr)
    y_pred = m.predict(X_te)
    print(name)
    print(classification_report(y_te, y_pred))

In [ ]:
from sklearn.model_selection import cross_val_score
cv_res = {}
for name, m in models.items():
    f1 = cross_val_score(m, X_scaled, labels, cv=5, scoring='f1_macro')
    cv_res[name] = f1.mean()
pd.Series(cv_res).sort_values(ascending=False)

In [ ]:
import joblib
best_name = pd.Series(cv_res).idxmax()
best_model = models[best_name]
joblib.dump(best_model, 'shchyolkovo_model.pkl')
joblib.dump(scaler, 'shchyolkovo_scaler.pkl')
print(f"Saved {best_name}")

In [ ]:
for c in sorted(h3_gdf['cluster'].unique()):
    print(f"\n~~~ Кластер {c} ~~~~")
    sub = h3_gdf[h3_gdf['cluster']==c]
    print("Средние значения:")
    print(sub[feat+['score']].mean())

## **Требования к отчету (Структуре блокнота)**



1. Описание выбранной территории и источников данных
2. Методика расчета экологических показателей с обоснованием выбора буферных зон и весовых коэффициентов
3. Сравнительный анализ результатов кластеризации с обоснованием выбора оптимального алгоритма
4. Карты распределения экологических факторов и результатов кластеризации
5. Подробная характеристика выявленных экологических зон (кластеров) с рекомендациями по их развитию
6. Анализ эффективности разработанных моделей классификации
7. Выводы об экологическом состоянии исследуемой территории и возможностях практического применения полученных результатов